# QLoRA Fine-Tuning Tutorial

This notebook demonstrates how to fine-tune a language model using QLoRA (Quantized Low-Rank Adaptation) for creating a thesaurus application.

## What is QLoRA?

QLoRA is an efficient fine-tuning method that combines:
- **4-bit quantization** of the base model
- **Low-rank adaptation** for parameter-efficient fine-tuning
- **Double quantization** to further reduce memory usage
- **NormalFloat (NF4)** data type optimized for normally distributed weights

This allows fine-tuning of large language models on consumer hardware with limited VRAM.

## Setup Environment

First, let's install the required packages:

In [ ]:
!pip install -q transformers>=4.30.0 peft>=0.4.0 accelerate>=0.20.0 bitsandbytes>=0.39.0 datasets>=2.12.0 trl>=0.4.7 scipy tqdm huggingface_hub

## Import Libraries

In [ ]:
import os
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType
)

## Create Sample Thesaurus Dataset

Let's create a small sample dataset for fine-tuning:

In [ ]:
# Sample thesaurus data
thesaurus_data = {
  "train": [
    {
      "word": "happy",
      "synonyms": ["joyful", "cheerful", "delighted", "pleased", "content"]
    },
    {
      "word": "sad",
      "synonyms": ["unhappy", "sorrowful", "dejected", "depressed", "downcast"]
    },
    {
      "word": "big",
      "synonyms": ["large", "huge", "enormous", "gigantic", "massive"]
    },
    {
      "word": "small",
      "synonyms": ["little", "tiny", "miniature", "compact", "diminutive"]
    },
    {
      "word": "fast",
      "synonyms": ["quick", "rapid", "swift", "speedy", "hasty"]
    }
  ],
  "test": [
    {
      "word": "poor",
      "synonyms": ["impoverished", "destitute", "needy", "penniless", "indigent"]
    },
    {
      "word": "hot",
      "synonyms": ["warm", "heated", "scorching", "burning", "fiery"]
    }
  ]
}

# Save to a JSON file
with open('thesaurus_dataset.json', 'w') as f:
    json.dump(thesaurus_data, f)

# Load the dataset
dataset = load_dataset('json', data_files='thesaurus_dataset.json')
dataset

## Select a Base Model

Choose a base model to fine-tune. For Colab, it's best to start with a smaller model:

In [ ]:
# Choose a smaller model for Colab
BASE_MODEL = "EleutherAI/pythia-1.4b"  # A smaller model that works well on Colab

# For more powerful hardware, you could use:
# BASE_MODEL = "meta-llama/Llama-2-7b-hf"  # Requires Hugging Face access token
# BASE_MODEL = "bigscience/bloom-1b7"      # Another good option
# BASE_MODEL = "facebook/opt-1.3b"         # Meta's OPT model

## Configure QLoRA

Set up the quantization and LoRA configurations:

In [ ]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load the base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Set padding token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## Prepare the Model for QLoRA Fine-Tuning

In [ ]:
# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                    # Rank of the update matrices
    lora_alpha=16,          # Scaling factor
    lora_dropout=0.05,      # Dropout probability
    bias="none",            # Don't train bias parameters
    task_type=TaskType.CAUSAL_LM,
    # Target the attention modules
    target_modules=["query_key_value"] if "pythia" in BASE_MODEL.lower() else ["q_proj", "k_proj", "v_proj", "o_proj"]
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters info
model.print_trainable_parameters()

## Prepare the Dataset for Training

Format the dataset for instruction tuning:

In [ ]:
def tokenize_function(examples):
    # Format for thesaurus data
    texts = [
        f"### Instruction: List synonyms for the word '{word}'\n\n### Response: {', '.join(synonyms)}"
        for word, synonyms in zip(examples['word'], examples['synonyms'])
    ]
    
    # Tokenize the texts
    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    
    # Set the labels to be the same as the inputs
    tokenized["labels"] = tokenized["input_ids"].clone()
    
    return tokenized

# Tokenize the dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

tokenized_dataset

## Set Up Training Arguments

In [ ]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir="./thesaurus-model-qlora",
    learning_rate=2e-4,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    save_steps=50,
    logging_steps=10,
    save_total_limit=3,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="tensorboard",
    load_best_model_at_end=True,
    gradient_checkpointing=True,  # Enable gradient checkpointing for memory efficiency
)

# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're not doing masked language modeling
)

## Initialize the Trainer and Start Training

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset.get("test", None),
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# Start training
print("Starting QLoRA fine-tuning...")
trainer.train()

## Save the Fine-Tuned Model

In [ ]:
# Save the final model
model.save_pretrained("./thesaurus-model-qlora")
tokenizer.save_pretrained("./thesaurus-model-qlora")
print("Model saved to ./thesaurus-model-qlora")

## Test the Fine-Tuned Model

In [ ]:
from peft import PeftModel, PeftConfig

# Load the configuration
config = PeftConfig.from_pretrained("./thesaurus-model-qlora")

# Load the base model with 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    load_in_4bit=True,
    device_map="auto"
)

# Load the LoRA adapter
fine_tuned_model = PeftModel.from_pretrained(base_model, "./thesaurus-model-qlora")

# Test words
test_words = ["beautiful", "intelligent", "strong", "weak"]

for word in test_words:
    # Prepare the prompt
    prompt = f"### Instruction: List synonyms for the word '{word}'\n\n### Response:"
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)
    
    # Generate the response
    print(f"\nGenerating synonyms for '{word}'...")
    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_length=100,
            temperature=0.7,
            num_beams=5,
            num_return_sequences=1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract the synonyms part
    response_parts = response.split("### Response:")
    if len(response_parts) > 1:
        synonyms = response_parts[1].strip()
    else:
        synonyms = response
    
    print(f"Synonyms for '{word}': {synonyms}")

## Optional: Merge the QLoRA Adapter with the Base Model

For more efficient inference, you can merge the adapter with the base model:

In [ ]:
# Merge the adapter with the base model
print("Merging adapter weights with base model...")
merged_model = fine_tuned_model.merge_and_unload()

# Save the merged model
output_path = "./thesaurus-model-merged"
print(f"Saving merged model to {output_path}...")
merged_model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

print("Model successfully merged and saved!")

## Download the Model for Use in Your Application

You can download the fine-tuned model to use in your application:

In [ ]:
# Create a zip file of the model
!zip -r thesaurus-model-qlora.zip ./thesaurus-model-qlora

# If you created a merged model
!zip -r thesaurus-model-merged.zip ./thesaurus-model-merged

# Download links will appear below
from google.colab import files
files.download('thesaurus-model-qlora.zip')
files.download('thesaurus-model-merged.zip')

## Conclusion

In this tutorial, you've learned how to:

1. Set up QLoRA for efficient fine-tuning
2. Prepare a thesaurus dataset
3. Fine-tune a language model with QLoRA
4. Test the fine-tuned model
5. Merge the adapter with the base model for efficient inference

You can now use this model in your thesaurus application to generate synonyms for words.

## References

- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [PEFT Library Documentation](https://huggingface.co/docs/peft/index)
- [BitsAndBytes Library](https://github.com/TimDettmers/bitsandbytes)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/index)